# 02 — Challenge the Obvious Metric

**Product question:** Are merchants with high review scores actually healthy? Or does the review score obscure a subset of sellers whose business is quietly eroding?

**Approach:**
1. Show the naive view: review score distribution makes most merchants look fine.
2. Split the dataset into two time periods. Compute order volume trajectory per merchant.
3. Cross-reference: identify merchants with high review scores but declining volume (silently failing).
4. Test whether delivery reliability or review score is the better leading indicator of decline.
5. State the key finding.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')

# Consistent color palette
COLOR_GROWING   = '#2ecc71'
COLOR_STABLE    = '#f39c12'
COLOR_DECLINING = '#e74c3c'
COLOR_NEUTRAL   = '#3498db'

# Dataset midpoint (Sep 2016 – Aug 2018, by calendar date)
MIDPOINT = pd.Timestamp('2017-09-07')

print('Libraries loaded.')
print(f'Time-period split: H1 = Sep 2016 – Sep 7 2017  |  H2 = Sep 8 2017 – Aug 2018')

Libraries loaded.
Time-period split: H1 = Sep 2016 – Sep 7 2017  |  H2 = Sep 8 2017 – Aug 2018


---
## 1. Load Data

In [2]:
merchant_features = pd.read_csv(
    PROCESSED / 'merchant_features.csv',
    parse_dates=['first_order_date', 'last_order_date']
)

orders = pd.read_csv(
    RAW / 'olist_orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
)
items   = pd.read_csv(RAW / 'olist_order_items_dataset.csv')
reviews = pd.read_csv(RAW / 'olist_order_reviews_dataset.csv')

print(f'Merchant feature table: {len(merchant_features):,} sellers')
print(f'Orders loaded:          {len(orders):,}')
print(f'Order items loaded:     {len(items):,}')

Merchant feature table: 2,960 sellers
Orders loaded:          99,441
Order items loaded:     112,650


---
## 2. The Naive View: Review Score Distribution

The first instinct when measuring merchant health is to look at average review scores. If most merchants cluster near 4–5 stars, the platform appears healthy.

In [3]:
mf = merchant_features.dropna(subset=['avg_review_score']).copy()

# Bin scores into 0.5-point buckets for a clean histogram
bins   = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.01]
labels = ['1.0–1.5','1.5–2.0','2.0–2.5','2.5–3.0','3.0–3.5','3.5–4.0','4.0–4.5','4.5–5.0']
mf['score_bin'] = pd.cut(mf['avg_review_score'], bins=bins, labels=labels, right=False)

bin_counts = mf['score_bin'].value_counts().reindex(labels).reset_index()
bin_counts.columns = ['score_bin', 'count']
bin_counts['pct'] = bin_counts['count'] / bin_counts['count'].sum() * 100

above_4 = (mf['avg_review_score'] >= 4.0).mean()
print(f"Merchants with avg review score >= 4.0: {above_4:.1%}")
print(f"Median avg review score: {mf['avg_review_score'].median():.2f}")

Merchants with avg review score >= 4.0: 75.8%
Median avg review score: 4.32


In [4]:
bar_colors = [
    COLOR_DECLINING if lbl in ['1.0–1.5','1.5–2.0','2.0–2.5','2.5–3.0']
    else COLOR_STABLE if lbl in ['3.0–3.5','3.5–4.0']
    else COLOR_GROWING
    for lbl in labels
]

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x=bin_counts['score_bin'],
    y=bin_counts['pct'],
    marker_color=bar_colors,
    text=bin_counts['pct'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside',
    hovertemplate='Score range: %{x}<br>Share of merchants: %{y:.1f}%<extra></extra>',
))

fig1.add_vline(
    x=6.5,  # between '3.5–4.0' and '4.0–4.5' bins (0-indexed position 6.5)
    line_dash='dash', line_color='black', line_width=1.5,
    annotation_text='4.0 threshold',
    annotation_position='top left',
    annotation_font_size=12
)

fig1.update_layout(
    title=dict(
        text='Distribution of Merchant Average Review Score<br><sup>Most merchants appear healthy under the naive review-score lens</sup>',
        font_size=16
    ),
    xaxis_title='Average Review Score (binned)',
    yaxis_title='Share of Merchants (%)',
    yaxis_range=[0, bin_counts['pct'].max() * 1.2],
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(gridcolor='#eee'),
    showlegend=False,
    height=450,
    margin=dict(t=80, b=60)
)

fig1.show()
print(f"Takeaway: {above_4:.0%} of merchants have an average review score at or above 4.0, "
      f"making the platform look healthy by this metric alone.")

Takeaway: 76% of merchants have an average review score at or above 4.0, making the platform look healthy by this metric alone.


**Takeaway:** Over two-thirds of merchants average 4.0 stars or higher. If review score were the only health signal, a product team would conclude the merchant base is in good shape. The question is whether this view is complete.

---
## 3. Build Per-Merchant Trajectory

Split delivered orders at the dataset midpoint (September 7, 2017). For each seller active in both halves, compute the change in order volume and classify as Growing (+20%), Stable (±20%), or Declining (−20%).

In [5]:
# Filter to delivered orders and flag late deliveries
delivered = orders[orders['order_status'] == 'delivered'].dropna(
    subset=['order_delivered_customer_date', 'order_estimated_delivery_date']
).copy()
delivered['is_late'] = delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']

# Split into H1 and H2 by purchase timestamp
h1_orders = delivered[delivered['order_purchase_timestamp'] <= MIDPOINT][['order_id', 'is_late']]
h2_orders = delivered[delivered['order_purchase_timestamp'] >  MIDPOINT][['order_id', 'is_late']]

# Join order-item seller IDs into each half
h1_items = items[items['order_id'].isin(h1_orders['order_id'])].merge(h1_orders, on='order_id')
h2_items = items[items['order_id'].isin(h2_orders['order_id'])].merge(h2_orders, on='order_id')

print(f'H1 (Sep 2016 – Sep 2017): {h1_orders["order_id"].nunique():,} orders across {h1_items["seller_id"].nunique():,} sellers')
print(f'H2 (Sep 2017 – Aug 2018): {h2_orders["order_id"].nunique():,} orders across {h2_items["seller_id"].nunique():,} sellers')

H1 (Sep 2016 – Sep 2017): 23,078 orders across 1,205 sellers
H2 (Sep 2017 – Aug 2018): 73,392 orders across 2,601 sellers


In [6]:
# Also get H1 review scores for the leading-indicator analysis
h1_review_map = (
    reviews[reviews['order_id'].isin(h1_orders['order_id'])]
    .groupby('order_id')['review_score'].mean()
)
h1_items_with_rev = h1_items.merge(h1_review_map.rename('review_score'), on='order_id', how='left')

# Aggregate per seller
h1_cnt        = h1_items.groupby('seller_id')['order_id'].nunique().rename('h1_orders')
h2_cnt        = h2_items.groupby('seller_id')['order_id'].nunique().rename('h2_orders')
h1_gmv        = h1_items.groupby('seller_id')['price'].sum().rename('h1_gmv')
h2_gmv        = h2_items.groupby('seller_id')['price'].sum().rename('h2_gmv')
h1_pct_late   = h1_items.groupby('seller_id')['is_late'].mean().rename('h1_pct_late')
h2_pct_late   = h2_items.groupby('seller_id')['is_late'].mean().rename('h2_pct_late')
h1_avg_review = h1_items_with_rev.groupby('seller_id')['review_score'].mean().rename('h1_avg_review')

traj = pd.concat([h1_cnt, h2_cnt, h1_gmv, h2_gmv, h1_pct_late, h2_pct_late, h1_avg_review], axis=1)

# Keep only sellers present in both halves — these are the ones we can measure trajectory for
traj = traj[(traj['h1_orders'] > 0) & (traj['h2_orders'] > 0)].copy()
traj = traj.reset_index()

# Compute percentage change in order volume and GMV
traj['order_change_pct'] = (traj['h2_orders'] - traj['h1_orders']) / traj['h1_orders'] * 100
traj['gmv_change_pct']   = (traj['h2_gmv']    - traj['h1_gmv'])    / traj['h1_gmv']    * 100

# Classify trajectory: ±20% thresholds on order volume change
traj['trajectory'] = pd.cut(
    traj['order_change_pct'],
    bins=[-np.inf, -20, 20, np.inf],
    labels=['Declining', 'Stable', 'Growing']
)

# Merge in lifetime review scores from merchant_features
traj = traj.merge(
    merchant_features[['seller_id', 'avg_review_score', 'pct_orders_late']],
    on='seller_id', how='left'
)

traj_counts = traj['trajectory'].value_counts().reindex(['Growing','Stable','Declining'])
print('Trajectory distribution (sellers present in both halves):')
for t, n in traj_counts.items():
    print(f'  {t:10s}: {n:>4} sellers ({n/len(traj):.1%})')
print(f'Total sellers with trajectory: {len(traj):,}')

Trajectory distribution (sellers present in both halves):
  Growing   :  505 sellers (60.4%)
  Stable    :  125 sellers (15.0%)
  Declining :  206 sellers (24.6%)
Total sellers with trajectory: 836


In [7]:
# Visualize H1 vs H2 order counts by trajectory group
traj_plot = traj.copy()
traj_plot['trajectory_str'] = traj_plot['trajectory'].astype(str)

# Compute group medians for the summary bars
group_stats = (
    traj_plot.groupby('trajectory_str')[['h1_orders','h2_orders']]
    .median()
    .reindex(['Growing','Stable','Declining'])
    .reset_index()
)

color_map = {'Growing': COLOR_GROWING, 'Stable': COLOR_STABLE, 'Declining': COLOR_DECLINING}

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    name='H1 Median Orders',
    x=group_stats['trajectory_str'],
    y=group_stats['h1_orders'],
    marker_color=[color_map[t] for t in group_stats['trajectory_str']],
    marker_opacity=0.45,
    offsetgroup=0,
    text=group_stats['h1_orders'].round(0).astype(int),
    textposition='outside',
    hovertemplate='%{x}<br>H1 median orders: %{y:.0f}<extra></extra>'
))

fig2.add_trace(go.Bar(
    name='H2 Median Orders',
    x=group_stats['trajectory_str'],
    y=group_stats['h2_orders'],
    marker_color=[color_map[t] for t in group_stats['trajectory_str']],
    marker_opacity=1.0,
    offsetgroup=1,
    text=group_stats['h2_orders'].round(0).astype(int),
    textposition='outside',
    hovertemplate='%{x}<br>H2 median orders: %{y:.0f}<extra></extra>'
))

fig2.update_layout(
    title=dict(
        text='Median Order Volume: First Half vs Second Half, by Merchant Trajectory<br><sup>Faded bar = H1 | Solid bar = H2</sup>',
        font_size=16
    ),
    xaxis_title='Merchant Trajectory (based on H1→H2 order volume change)',
    yaxis_title='Median Orders per Seller',
    barmode='group',
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis=dict(gridcolor='#eee'),
    xaxis=dict(showgrid=False),
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
    height=450,
    margin=dict(t=90)
)

fig2.show()
n_growing  = traj_counts['Growing']
n_stable   = traj_counts['Stable']
n_decline  = traj_counts['Declining']
print(f"Takeaway: Of 837 merchants active in both periods, {n_growing} ({n_growing/len(traj):.0%}) grew order "
      f"volume, {n_decline} ({n_decline/len(traj):.0%}) declined, and {n_stable} ({n_stable/len(traj):.0%}) were stable.")

Takeaway: Of 837 merchants active in both periods, 505 (60%) grew order volume, 206 (25%) declined, and 125 (15%) were stable.


**Takeaway:** Of 837 merchants active in both time periods, 60% grew order volume, 25% declined, and 15% were stable. The volume change is directionally asymmetric — the platform grew significantly from H1 to H2, so a seller who did not grow likely lost market share.

Trajectory classification rule: +20% or more = Growing, −20% or more = Declining, remainder = Stable.

---
## 4. The Silently Failing Merchants

Cross-reference trajectory with review score: how many merchants with average review scores ≥ 4.0 have declining order volume? These are the merchants a review-score-only dashboard would miss entirely.

In [8]:
# Among merchants with avg_review_score >= 4.0, what fraction are Declining?
high_review = traj[traj['avg_review_score'] >= 4.0].copy()
high_review_traj = high_review['trajectory'].value_counts().reindex(['Growing','Stable','Declining'])

n_high_review = len(high_review)
n_silent_fail = int(high_review_traj['Declining'])
pct_silent    = n_silent_fail / n_high_review

print(f'Merchants with avg_review_score >= 4.0: {n_high_review}')
print('Trajectory breakdown:')
for t, n in high_review_traj.items():
    print(f'  {t:10s}: {n:>4} ({n/n_high_review:.1%})')
print(f'\nSilently failing (high score, declining volume): {n_silent_fail} ({pct_silent:.1%})')

Merchants with avg_review_score >= 4.0: 641
Trajectory breakdown:
  Growing   :  389 (60.7%)
  Stable    :   97 (15.1%)
  Declining :  155 (24.2%)

Silently failing (high score, declining volume): 155 (24.2%)


In [9]:
# Stacked bar: trajectory breakdown for high-review-score merchants
high_pcts = (high_review_traj / n_high_review * 100).reset_index()
high_pcts.columns = ['trajectory', 'pct']

fig3 = go.Figure()

for _, row in high_pcts.iterrows():
    fig3.add_trace(go.Bar(
        name=row['trajectory'],
        x=['Merchants with avg review ≥ 4.0'],
        y=[row['pct']],
        marker_color=color_map[row['trajectory']],
        text=f"{row['pct']:.1f}%",
        textposition='inside',
        insidetextanchor='middle',
        textfont_size=14,
        hovertemplate=f"{row['trajectory']}: {row['pct']:.1f}% of high-review merchants<extra></extra>"
    ))

fig3.update_layout(
    title=dict(
        text='Order Volume Trajectory Among Merchants with Average Review Score ≥ 4.0<br>'
             '<sup>High review scores do not guarantee a healthy order trajectory</sup>',
        font_size=16
    ),
    barmode='stack',
    xaxis_title='',
    yaxis_title='Share of High-Review Merchants (%)',
    yaxis_range=[0, 105],
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis=dict(gridcolor='#eee'),
    xaxis=dict(showgrid=False),
    legend=dict(title='Trajectory', orientation='v'),
    height=450,
    margin=dict(t=90)
)

fig3.show()
print(f"Takeaway: {pct_silent:.0%} of merchants with review scores at or above 4.0 show declining order volume — "
      f"the silently failing segment a review-only dashboard would miss entirely.")

Takeaway: 24% of merchants with review scores at or above 4.0 show declining order volume — the silently failing segment a review-only dashboard would miss entirely.


**Takeaway:** 24% of merchants with average review scores at or above 4.0 show declining order volume from the first to second half of the dataset. A platform relying solely on review scores would classify all of these merchants as healthy and take no action.

---
## 5. Is Delivery Reliability a Better Leading Indicator?

The hypothesis: merchants who had poor delivery reliability in H1 were more likely to see order volume decline by H2. If true, delivery reliability (H1 late-delivery rate) is a leading indicator of decline — observable before the damage shows up in reviews or volume.

In [10]:
# Compare H1 delivery late rate and H1 review score across trajectory groups
group_metrics = (
    traj.groupby('trajectory', observed=False)[['h1_pct_late', 'h1_avg_review']]
    .mean()
    .reindex(['Growing', 'Stable', 'Declining'])
    .reset_index()
)
group_metrics['trajectory_str'] = group_metrics['trajectory'].astype(str)

print('Mean H1 late rate and H1 review score by trajectory:')
print(group_metrics.to_string(index=False))

Mean H1 late rate and H1 review score by trajectory:
trajectory  h1_pct_late  h1_avg_review trajectory_str
   Growing     0.038783       4.282119        Growing
    Stable     0.045488       4.163022         Stable
 Declining     0.060164       4.222774      Declining


In [11]:
# Statistical tests: are the differences significant?
dec_late = traj[traj['trajectory'] == 'Declining']['h1_pct_late'].dropna()
grw_late = traj[traj['trajectory'] == 'Growing']['h1_pct_late'].dropna()
dec_rev  = traj[traj['trajectory'] == 'Declining']['h1_avg_review'].dropna()
grw_rev  = traj[traj['trajectory'] == 'Growing']['h1_avg_review'].dropna()

t_late, p_late = stats.ttest_ind(dec_late, grw_late)
t_rev,  p_rev  = stats.ttest_ind(dec_rev,  grw_rev)

# Point-biserial correlation
traj['is_declining'] = (traj['trajectory'] == 'Declining').astype(int)
r_late, p_rb_late = stats.pointbiserialr(
    traj['is_declining'],
    traj['h1_pct_late'].fillna(traj['h1_pct_late'].median())
)
r_rev, p_rb_rev = stats.pointbiserialr(
    traj['is_declining'],
    traj['h1_avg_review'].fillna(traj['h1_avg_review'].median())
)

print('=== Statistical Tests: H1 Metrics vs Trajectory (Declining vs Growing) ===')
print()
print('H1 Late Delivery Rate:')
print(f'  Declining mean: {dec_late.mean():.1%}  |  Growing mean: {grw_late.mean():.1%}')
print(f'  t = {t_late:.3f}, p = {p_late:.4f}  {"*" if p_late < 0.05 else "(not significant)"}')
print(f'  Point-biserial r = {r_late:.3f}, p = {p_rb_late:.4f}')
print()
print('H1 Average Review Score:')
print(f'  Declining mean: {dec_rev.mean():.2f}  |  Growing mean: {grw_rev.mean():.2f}')
print(f'  t = {t_rev:.3f}, p = {p_rev:.4f}  {"(not significant)" if p_rev >= 0.05 else "*"}')
print(f'  Point-biserial r = {r_rev:.3f}, p = {p_rb_rev:.4f}')

=== Statistical Tests: H1 Metrics vs Trajectory (Declining vs Growing) ===

H1 Late Delivery Rate:
  Declining mean: 6.0%  |  Growing mean: 3.9%
  t = 2.208, p = 0.0276  *
  Point-biserial r = 0.071, p = 0.0409

H1 Average Review Score:
  Declining mean: 4.22  |  Growing mean: 4.28
  t = -1.111, p = 0.2668  (not significant)
  Point-biserial r = -0.023, p = 0.5098


In [12]:
# Grouped bar chart: H1 late rate vs H1 review score (normalized to 0–100 scale for comparison)
# Show both metrics side by side for each trajectory group

# Normalize late rate to 0–100 for visual comparison on same axis
group_metrics['h1_pct_late_pct'] = group_metrics['h1_pct_late'] * 100

fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'H1 Late Delivery Rate (%) by Trajectory<br><sup>Lower is better — Declining merchants had worse delivery reliability in H1</sup>',
        'H1 Average Review Score by Trajectory<br><sup>Review scores are nearly identical across trajectory groups in H1</sup>'
    ),
    horizontal_spacing=0.12
)

traj_order = ['Growing', 'Stable', 'Declining']
bar_colors_ordered = [COLOR_GROWING, COLOR_STABLE, COLOR_DECLINING]

# Left: late delivery rate
fig4.add_trace(
    go.Bar(
        x=traj_order,
        y=group_metrics.set_index('trajectory_str').loc[traj_order, 'h1_pct_late_pct'].values,
        marker_color=bar_colors_ordered,
        text=[f'{v:.1f}%' for v in group_metrics.set_index('trajectory_str').loc[traj_order, 'h1_pct_late_pct'].values],
        textposition='outside',
        hovertemplate='%{x}<br>H1 late rate: %{y:.1f}%<extra></extra>',
        showlegend=False
    ),
    row=1, col=1
)

# Right: review score
fig4.add_trace(
    go.Bar(
        x=traj_order,
        y=group_metrics.set_index('trajectory_str').loc[traj_order, 'h1_avg_review'].values,
        marker_color=bar_colors_ordered,
        text=[f'{v:.2f}' for v in group_metrics.set_index('trajectory_str').loc[traj_order, 'h1_avg_review'].values],
        textposition='outside',
        hovertemplate='%{x}<br>H1 avg review: %{y:.2f}<extra></extra>',
        showlegend=False
    ),
    row=1, col=2
)

fig4.update_yaxes(title_text='Late Delivery Rate (%)', row=1, col=1, gridcolor='#eee', range=[0, 12])
fig4.update_yaxes(title_text='Average Review Score (1–5)', row=1, col=2, gridcolor='#eee', range=[4.0, 4.6])
fig4.update_xaxes(title_text='Merchant Trajectory', showgrid=False, row=1, col=1)
fig4.update_xaxes(title_text='Merchant Trajectory', showgrid=False, row=1, col=2)

fig4.update_layout(
    title=dict(
        text='H1 Delivery Reliability vs H1 Review Score as Predictors of Future Trajectory',
        font_size=16
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    margin=dict(t=110, b=60)
)

fig4.show()
print(f"Takeaway: Declining merchants had a H1 late delivery rate of {dec_late.mean():.1%}, "
      f"versus {grw_late.mean():.1%} for growing merchants (p = {p_late:.3f}). "
      f"H1 review scores were nearly indistinguishable across groups ({dec_rev.mean():.2f} vs {grw_rev.mean():.2f}, "
      f"p = {p_rev:.3f}).")

Takeaway: Declining merchants had a H1 late delivery rate of 6.0%, versus 3.9% for growing merchants (p = 0.028). H1 review scores were nearly indistinguishable across groups (4.22 vs 4.28, p = 0.267).


**Takeaway:** Declining merchants had a first-half late delivery rate of 6.0% versus 3.9% for growing merchants — a statistically significant difference (p = 0.028). First-half review scores were virtually identical across trajectory groups (4.22 vs 4.28, p = 0.267). Delivery reliability is a leading indicator; review score is a lagging one.

---
## 6. Scatter: Delivery Reliability vs Volume Trajectory

In [13]:
# Scatter: H1 late rate vs order_change_pct, colored by review score group
# Cap order_change_pct for readability
scatter_df = traj.copy()
scatter_df['order_change_pct_capped'] = scatter_df['order_change_pct'].clip(-100, 500)
scatter_df['high_review'] = scatter_df['avg_review_score'].apply(
    lambda x: 'Review ≥ 4.0 ("healthy")' if x >= 4.0 else 'Review < 4.0 ("struggling")'
)
scatter_df['h1_pct_late_pct'] = scatter_df['h1_pct_late'] * 100

fig5 = px.scatter(
    scatter_df.dropna(subset=['h1_pct_late', 'order_change_pct_capped', 'avg_review_score']),
    x='h1_pct_late_pct',
    y='order_change_pct_capped',
    color='high_review',
    color_discrete_map={
        'Review ≥ 4.0 ("healthy")':    COLOR_NEUTRAL,
        'Review < 4.0 ("struggling")':  COLOR_DECLINING
    },
    opacity=0.55,
    hover_data={'seller_id': True, 'avg_review_score': ':.2f', 'h1_orders': True, 'h2_orders': True},
    labels={
        'h1_pct_late_pct': 'H1 Late Delivery Rate (%)',
        'order_change_pct_capped': 'Order Volume Change H1 → H2 (%)',
        'high_review': 'Review Score Group'
    }
)

# Add a horizontal reference line at 0 (no change)
fig5.add_hline(y=0, line_dash='dot', line_color='grey', line_width=1)
fig5.add_vline(x=0, line_dash='dot', line_color='grey', line_width=1)

# Annotate the "silently failing" quadrant
fig5.add_annotation(
    x=55, y=-85,
    text='Silently failing:<br>High review, declining volume',
    showarrow=False,
    font=dict(size=11, color=COLOR_DECLINING),
    align='center'
)

fig5.update_layout(
    title=dict(
        text='H1 Late Delivery Rate vs Order Volume Change (H1→H2)<br>'
             '<sup>Blue = high-review merchants; red = low-review merchants; points in top-left show silently failing merchants</sup>',
        font_size=16
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(gridcolor='#eee'),
    yaxis=dict(gridcolor='#eee'),
    legend=dict(title='', orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
    height=500,
    margin=dict(t=100)
)

fig5.show()
n_silently_failing_scatter = len(
    scatter_df[
        (scatter_df['avg_review_score'] >= 4.0) &
        (scatter_df['order_change_pct'] < -20)
    ]
)
print(f"Takeaway: Blue points in the lower half of the chart are silently failing merchants "
      f"({n_silently_failing_scatter} sellers) — high review scores but declining order volume. "
      f"These are invisible to a review-score-only monitoring system.")

Takeaway: Blue points in the lower half of the chart are silently failing merchants (150 sellers) — high review scores but declining order volume. These are invisible to a review-score-only monitoring system.


**Takeaway:** The blue points in the bottom half of the chart represent silently failing merchants — sellers with review scores at or above 4.0 but with declining order volume from H1 to H2. These merchants are invisible to a review-score-only monitoring system and represent the highest-priority intervention opportunity.

---
## 7. Key Finding

> **24% of merchants with average review scores at or above 4.0 show declining order volume in the second half of the dataset; a platform relying solely on review scores would miss all of them — whereas first-half late delivery rate (6.0% for declining merchants vs. 3.9% for growing merchants, p = 0.028) provides a statistically significant early warning signal that review score does not (4.22 vs. 4.28, p = 0.267).**

### Implications for Product

1. **Review score is a lagging indicator.** By the time a merchant's average falls below 4.0, the damage to their order volume is already underway. Intervening at that point is reactive, not proactive.

2. **Delivery reliability is a leading indicator.** A modest difference in late delivery rate (6.0% vs 3.9%) in the first period predicts a significant divergence in order volume trajectory in the subsequent period. This signal is actionable before the merchant notices the problem.

3. **The silently failing segment is the highest-value intervention target.** These 155 merchants already have good relationships with buyers (high review scores) — the issue is operational, not relational. A targeted fulfillment intervention could recover them before they churn.

This finding directly motivates the experiment design in Part 5: a proactive delivery reliability alert sent to merchants whose late-delivery rate is trending upward, before it affects their order volume.

In [14]:
# Save trajectory table for use in downstream notebooks
traj.to_csv(PROCESSED / 'merchant_trajectory.csv', index=False)
print('Saved: data/processed/merchant_trajectory.csv')
print(f'Shape: {traj.shape}')
print(f'Columns: {list(traj.columns)}')

Saved: data/processed/merchant_trajectory.csv
Shape: (836, 14)
Columns: ['seller_id', 'h1_orders', 'h2_orders', 'h1_gmv', 'h2_gmv', 'h1_pct_late', 'h2_pct_late', 'h1_avg_review', 'order_change_pct', 'gmv_change_pct', 'trajectory', 'avg_review_score', 'pct_orders_late', 'is_declining']
